<a href="https://colab.research.google.com/github/Valementajat/PageRank_colab/blob/master/pageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
''' pip install kaggle '''

In [3]:
# install PySpark
!apt-get update # Update apt-get repository.
!apt-get install openjdk-8-jdk-headless -qq > /dev/null # Install Java.
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz # Download Apache Sparks.
!tar xf spark-3.1.1-bin-hadoop3.2.tgz # Unzip the tgz file.
!pip install -q findspark # Install findspark. Adds PySpark to the System path during runtime.


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,615 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,160 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd6

In [4]:
import os
os.environ['KAGGLE_USERNAME'] = "XXXXX"
os.environ['KAGGLE_KEY'] = "XXXXX"

# Set environment variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"

dataset_zip = "new-york-times-articles-comments-2020.zip"
if not os.path.exists(dataset_zip):
  !kaggle datasets download -d  benjaminawd/new-york-times-articles-comments-2020




Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 1.94G/1.95G [00:22<00:00, 114MB/s] 
100% 1.95G/1.95G [00:22<00:00, 93.0MB/s]


In [5]:
# Initialize findspark
import findspark
findspark.init()

# Create a PySpark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

In [ ]:
# Unzip the file independenty
articles = "nyt-articles-2020.csv"
comments = "nyt-comments-2020.csv"

if not os.path.exists(articles) and not os.path.exists(comments):
  !unzip new-york-times-articles-comments-2020.zip

Archive:  new-york-times-articles-comments-2020.zip
  inflating: nyt-articles-2020.csv   
  inflating: nyt-comments-2020.csv   
  inflating: nyt-comments-part0.csv  
  inflating: nyt-comments-part1.csv  
  inflating: nyt-comments-part2.csv  
  inflating: nyt-comments-part3.csv  
  inflating: nyt-comments-part4.csv  
  inflating: nyt-comments-part5.csv  
  inflating: nyt-comments-part6.csv  
  inflating: nyt-comments-part7.csv  
  inflating: nyt-comments-part8.csv  
  inflating: nyt-comments-part9.csv  
  inflating: test.csv                
  inflating: train.csv               


In [ ]:
# Add the wanted

In [ ]:
import numpy as np
import pandas as pd

# importing random for making a subsample of the dataset
import random

from collections import defaultdict, Counter
import gc

In [ ]:
# Lets initialize a sample freq at the start so it's easy to spot
# Sample_fraq is the sample of articles we're concidering, you can set this between 0.01 and 1.0
sample_frac = 0.2

In [ ]:
chunk_size = 10000

# Using a set to avoid duplicates
ids = set()


# To guard against the memory we load the csv in chuncks and process it right away
for article_chunk in pd.read_csv("nyt-articles-2020.csv", chunksize=chunk_size):
   for article_id in article_chunk["uniqueID"]:
      ids.add(article_id)

n = len(ids)

In [ ]:
# Lets create the dataset we want to work from the articles
sample_size = int(n * sample_frac)

# Adding a seed to make the sampling reproducible
random.seed(10)
sample_ids = random.sample(list(ids), sample_size)
n = len(sample_ids)
# Maybe this will be a bit more efficient to work with than a full on URL?
id_to_i = {article_id: i for i, article_id in enumerate(sample_ids)}



In [ ]:
# Creating an empty adjancency matrix
adjencencyMatrix =  np.zeros((n, n ), dtype=float)



In [ ]:
# TO avoid blowing up the memory usage, we load the
# comments in chucks
chunk_size = 10000

# First part of the mapReduce, we create the key value pairs
KeyValuePairs = []

for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size):
  # Start the MapReduce by creating key-value pairs
  for index, row in chunk.iterrows():
    if row['articleID'] in id_to_i:
      # Currently creating this like "article -> user", at now it's okay i guess
      KeyValuePairs.append((id_to_i[row["articleID"]], row["userID"]))

del chunk
del chunk_size

gc.collect()



0

In [ ]:
# Lets group the KeyValuePairs by the key
KeyValueGroup = defaultdict(set)
for article_idx, user_id in KeyValuePairs:
    # but
    KeyValueGroup[article_idx].add(user_id)


In [ ]:
# Then the final reduce so we can use this
connectionPairs = []

items = list(KeyValueGroup.items())

# This is a possibly a problem?
# I've created the groups with article [uids], but it would be more efficient
# with uid [articles] ?

# I'm checking items against other values in the same set and looking if there
# are two unique intersecting user pairs so we can say that there
for a_idx, users_a in items:
    for b_idx, users_b in items:
        if a_idx != b_idx:
            if len(users_a.intersection(users_b)) >= 2:
                connectionPairs.append((a_idx, b_idx))


In [ ]:
for i in range(10):
  print(connectionPairs[i])

(14397, 12494)
(14397, 14713)
(14397, 6375)
(14397, 9595)
(14397, 15240)
(14397, 14689)
(14397, 4655)
(14397, 1183)
(14397, 2588)
(14397, 1457)


In [ ]:
# degree is practically how many times that page is seen in the connections
degree = Counter()
for i, j in connectionPairs:
    degree[i] += 1


for i, j in connectionPairs:
    adjencencyMatrix[i, j] = 1.0 / degree[i]


In [ ]:
#We have a problem, dangling nodes, so this is to uniformally
# set the propability to go from a page to every page when calculating the rank

row_sums = adjencencyMatrix.sum(axis=1)
dangling = (row_sums == 0)


# replace each dangling row with uniform distribution
adjencencyMatrix[dangling, :] = 1.0 / n

In [ ]:
# So we have our stochastic adjencencyMatrix or connection matrix

# Without any dangling pages (or nodes)

# Lets run the power iteration

# This is the formula to "teleport", during the power iteration.
# as in, do we take the next page from the page's links or do we take one from
# all of the pages
# P = βM + (1 - β)1/N

beta = 0.85 # example size is between 0.8 - 0.9
n = len(adjencencyMatrix)

#now this is the pageRank transition matrix or the google matrix
P = beta * adjencencyMatrix + (1 - beta) / n

In [ ]:
# lets initialize the rank vector r^(0) which has every rank as 1/n
r = np.ones(n) / n

In [ ]:
# Now we calculat the new ranks using the transition matrix and the old rank
# r^new =  r^old · A
for i in range(100):
  # python's matrix manipulation to multiply matrix with another matrix
    r = r @ P

In [ ]:
# This is just a sanity check
print(r.sum())

0.9999999999999766


In [ ]:
# Top 10 indices by rank (largest first)
top_k = 10
top_idx = np.argsort(r)[-top_k:][::-1]

print("Top ranks:")
for idx in top_idx:
    print(idx, r[idx])

Top ranks:
10928 0.00030979547372165595
1649 0.00025811016395415535
1267 0.00024623683448383153
14265 0.00024577547537928764
11101 0.00023959482952740137
14202 0.0002345407473430717
2779 0.0002336741920158343
10396 0.00023210545802228728
3491 0.00023007565615243308
15642 0.0002279752851877883


In [ ]:
%whos

Variable           Type           Data/Info
-------------------------------------------
Counter            type           <class 'collections.Counter'>
KeyValueGroup      defaultdict    defaultdict(<class 'set'><...>11771, 34452477, 86527}})
KeyValuePairs      list           n=4986461
P                  ndarray        16787x16787: 281803369 elems, type `float64`, 2254426952 bytes (2149.989082336426 Mb)
a_idx              int            8
adjencencyMatrix   ndarray        16787x16787: 281803369 elems, type `float64`, 2254426952 bytes (2149.989082336426 Mb)
article_chunk      DataFrame              newsdesk       se<...>n[6787 rows x 11 columns]
article_id         str            nyt://article/12048b2b-62<...>e3-5bed-8c77-483a4299f465
article_idx        int            8
articles           str            nyt-articles-2020.csv
b_idx              int            8
beta               float          0.85
comments           str            nyt-comments-2020.csv
connectionPairs    list           n